# In class Week 12 Tutorial

In [ ]:
import enn554.pv as pv
import enn554.constants as const
import numpy as np
import matplotlib.pyplot as plt

## Exercise 1

Part a)

In [ ]:
G_ref,T_ref = 1000, 273.15 + 25
I_mp_ref, δI_mp = 3.96, -0.00154
V_mp_ref, δV_mp = 33.68, -0.15358
NOCT, G_NOCT = 316.85, 800
G,T_amb = 950, 40+273.15

Compute new cell temperature

In [ ]:
Tc = T_amb + (NOCT-293.15)/G_NOCT * G
print(f'Tc={Tc:.2f}K, {Tc-273.15:.2f}C')

Get power at new condition

In [ ]:
I_mp = I_mp_ref + δI_mp * (Tc-T_ref)
V_mp = V_mp_ref + δV_mp * (Tc-T_ref)
P_mp = I_mp * V_mp
print(f"New MPP: ({V_mp:.2f}V, {I_mp:.2f}A)")
print(f"New Power: {P_mp:.2f}W")

b)

In [ ]:
module = pv.DeSotoModel(Ns=72,Isc_ref=4.37,Voc_ref=42.93,Imp_ref=I_mp_ref,
                        Vmp_ref=V_mp_ref,delta_Isc=0.00175,delta_Voc=-0.15237)
vmp,imp,pmp = module.mpp(G=G,T_C = Tc-273.15)

print(f"New MPP: ({vmp:.2f}V, {imp:.2f}A)")
print(f"New Power: {pmp:.2f}W")

In [ ]:
ax = module.plot_iv()
module.plot_iv(ax=ax,G=G,T_C = Tc-273.15)
ax.plot(vmp,imp,'r*',markersize=10,label='MPP')
ax.legend()

## Exercise 2

Let $n$ be the number of modules. The shaded voltage is
\begin{align}
    \Delta V \approx \frac{V}{n} + (I-I_{L,shaded})R_p
\end{align}

In [ ]:
Tc = 273.17+25 + (NOCT-293.15)/800 * 800
Tc_C = Tc - 273.15
module2 = pv.DeSotoModel(Ns=72,Isc_ref=4.37,Voc_ref=42.93,Imp_ref=I_mp_ref,
                        Vmp_ref=V_mp_ref,delta_Isc=0.00175,delta_Voc=-0.15237)

# sunny module
v,i,p = module2.iv_curve(G=800,T_C = Tc_C,n_points=2000)
idx_mpp = np.argmax(p)
P0 = 4*p[idx_mpp] # all modules with G = 800

# shading effect
ISH = module2.current(V=0,G=120,T_C=Tc_C)
_,_,_,_,Rsh = module2.translate(G=120,T_C = Tc_C)
V_shaded_module = np.array([module2.voltage(I=current,G=120,T_C = Tc_C) for current in i])
VSH = 3*v
VSH[i<=ISH] += V_shaded_module[i<=ISH]
VSH[i>ISH] -= v[i>ISH]/module2.Ns + (i[i>ISH]-ISH)*Rsh
idx_mpp_SH = np.argmax(i*VSH)
PSH = VSH[idx_mpp_SH]*i[idx_mpp_SH]


In [ ]:
fig,ax = plt.subplots()
ax.plot(4*v,i,label=f'No shading (P={P0:.1f}W)')
ax.plot(4*v[idx_mpp],i[idx_mpp],'r*',markersize=10,label='MPP (No shading)')
ax.plot(VSH,i,label=f'With shading (P={PSH:.1f}W)')
ax.plot(VSH[idx_mpp_SH],i[idx_mpp_SH],'r*',markersize=10,label='MPP (shading shading)')
ax.set_xlim((0,160))
ax.set_ylim((0,4))
ax.legend()


In [ ]:
poa_loss_fraction = 1 - (3*800+120)/4/800
output_loss_fraction = 1-PSH/P0
print(f'Lost {poa_loss_fraction*100:.2f}% of the INCIDENT Power')
print(f'Lost {output_loss_fraction*100:.2f}% of the Output Power')